**Task 7 ML***

In [13]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import re

def parse_log_file(file_path):
    """Raw log file ko parse karke DataFrame banata hai"""
    data = []
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            # Extract IP address
            ip_match = re.search(r'(\d{1,3}(?:\.\d{1,3}){3})', line)
            ip = ip_match.group(1) if ip_match else '127.0.0.1'

            # Extract HTTP Status Code
            status_match = re.search(r'\s([1-5]\d{2})\s', line)
            status = int(status_match.group(1)) if status_match else 200

            # Extract Payload Size
            size_match = re.search(r'\s(\d+)\s*$', line.strip())
            size = int(size_match.group(1)) if size_match else 0

            # Extract HTTP Method
            method_match = re.search(r'"(GET|POST|PUT|DELETE|HEAD)', line)
            method = method_match.group(1) if method_match else 'GET'

            data.append({
                'ip': ip,
                'method': method,
                'status': status,
                'payload_size': size,
                'raw_log': line.strip()
            })

    return pd.DataFrame(data)

def extract_features(df):
    """IPs ke basis par behavioral features aggregate karta hai"""
    features = df.groupby('ip').agg(
        total_requests=('ip', 'count'),
        error_404_count=('status', lambda x: (x == 404).sum()),
        server_error_count=('status', lambda x: (x >= 500).sum()),
        avg_payload_size=('payload_size', 'mean'),
        max_payload_size=('payload_size', 'max')
    ).reset_index()

    return features

def detect_anomalies(features_df):
    """Isolation Forest model train karke anomalies detect karta hai"""
    X = features_df[['total_requests', 'error_404_count', 'server_error_count', 'avg_payload_size', 'max_payload_size']]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Isolation Forest model
    model = IsolationForest(contamination=0.05, random_state=42)
    features_df['anomaly'] = model.fit_predict(X_scaled)

    return features_df

# Execution Pipeline
if __name__ == "__main__":
    log_file = "/content/logfiles.log"

    print("[INFO] Parsing raw server logs...")
    raw_df = parse_log_file(log_file)
    print(f"[INFO] Successfully parsed {len(raw_df)} log lines.")

    print("[INFO] Extracting technical features per IP...")
    features = extract_features(raw_df)

    print("[INFO] Running Unsupervised Anomaly Detection (Isolation Forest)...")
    analyzed_df = detect_anomalies(features)

    threats = analyzed_df[analyzed_df['anomaly'] == -1]
    print(f"\n[ALERT] Detected {len(threats)} anomalous IPs!")
    print(threats)

[INFO] Parsing raw server logs...
[INFO] Successfully parsed 259761 log lines.
[INFO] Extracting technical features per IP...
[INFO] Running Unsupervised Anomaly Detection (Isolation Forest)...

[ALERT] Detected 12928 anomalous IPs!
                   ip  total_requests  error_404_count  server_error_count  \
7        1.10.117.118               1                1                   0   
85       1.115.58.167               1                1                   0   
110      1.12.184.149               1                1                   0   
152      1.128.46.123               1                0                   1   
159      1.13.187.171               1                1                   0   
...               ...             ...              ...                 ...   
259685   99.82.64.208               1                0                   1   
259701  99.86.128.239               1                0                   1   
259720  99.90.221.120               1                0           